In [1]:
! pip install selenium
! pip install webdriver-manager
! pip install pandas


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

URL = "https://www.themoviedb.org/movie"

def create_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    return driver


def test_click_sort():
    driver = create_driver()
    driver.get(URL)
    time.sleep(2)

    # STEP A: "정렬" 텍스트가 있는 h2 찾기 → 부모 div.name 클릭
    sort_toggle = driver.find_element(By.XPATH, "//h2[contains(text(),'정렬')]/parent::div")
#    print("정렬 패널 찾음! 클릭합니다...")
    sort_toggle.click()
    print("정렬 패널 열림!")

    time.sleep(1)

    # STEP B: 드롭다운 select 버튼 클릭
    dropdown_button = driver.find_element(By.XPATH, "(//button[@aria-label='select'])[1]")
#    print("드롭다운 버튼 찾음! 클릭합니다...")
    dropdown_button.click()
    print("드롭다운 열림!")

 # STEP C: option 직접 선택
    print("STEP C: 평점 내림차순 선택 중...")

    rating_li = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable(
        (By.XPATH, "//ul[@role='listbox']//li[.//text()[contains(., '평점')]]")
    )
    )

    print("평점 내림차순 옵션 찾음 → 클릭!")
    rating_li.click()

    print("평점 내림차순 선택 완료!")

     # STEP D: 개봉일 날짜 설정
    print("STEP D: 날짜 입력 시작...")

    # 1) '개봉일로 검색' 체크박스 클릭 (필수!)
    release_checkbox = driver.find_element(By.XPATH, "//input[@id='all_releases']")
    if not release_checkbox.is_selected():
        release_checkbox.click()
    print("✔ 개봉일 검색 체크박스 활성화")

    time.sleep(1)

    # 2) 첫 번째 날짜 입력창 (검색날짜)
    date_from = driver.find_element(By.XPATH, "(//input[@data-role='datepicker'])[1]")

    date_from.clear()
    date_from.send_keys("2025-01-01")
    print("✔ 검색날짜 입력 완료: 2025-01-01")

    time.sleep(0.5)

    # 3) 두 번째 날짜 입력창 (부터)
    date_to = driver.find_element(By.XPATH, "(//input[@data-role='datepicker'])[2]")

    date_to.clear()
    date_to.send_keys("2025-03-31")
    print("✔ 검색날짜(끝) 입력 완료: 2025-03-31")

    time.sleep(1)

    # 4) 입력 확정 위해 바깥 아무 곳 클릭
    driver.find_element(By.TAG_NAME, "body").click()
    print("✔ 날짜 입력 확정 완료!")

    # STEP E: 언어 한국어 선택
    print("STEP E: 언어 한국어 선택 시작...")

    print("STEP E(1): 언어 드롭다운 열기 시도...")

    # 1) '언어' 라벨 기준으로 드롭다운 버튼 찾기
    language_button = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((
            By.XPATH,
            "//h3[contains(text(),'언어')]/following::button[@aria-label='select'][1]"
        ))
    )

    # 화면에 보이도록 스크롤
    driver.execute_script("arguments[0].scrollIntoView(true);", language_button)
    time.sleep(0.3)

    # 클릭
    language_button.click()
    print("✔ STEP E(1): 언어 드롭다운 열림 완료!")


    # 2) 나타난 <ul role='listbox'> 내부에서 '한국어' 클릭
    korean_item = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable(
            (By.XPATH, "//ul[@role='listbox']//li[.//text()[contains(., '한국어')]]")
        )
    )

    print("✔ 한국어 항목 발견 → 클릭!")
    korean_item.click()

    print("✔ 언어 ‘한국어’ UI 클릭으로 선택 완료!")

    from selenium.webdriver.common.action_chains import ActionChains

    print("STEP F: 슬라이더 핸들 드래그 시작...")

    # 상영시간 섹션을 기준으로 슬라이더 영역 가져오기
    runtime_section = driver.find_element(
        By.XPATH,
        "//h3[contains(text(),'상영시간')]/following-sibling::div[contains(@class,'k-slider')]"
    )

    # 핸들과 트랙 선택 (상영시간 전용)
    left_handle = runtime_section.find_elements(By.CSS_SELECTOR, ".k-draghandle")[0]
    track = runtime_section.find_element(By.CSS_SELECTOR, ".k-slider-track")

    # 트랙 길이(px)
    track_width = track.size['width']

    # 이동해야 할 거리 계산 (0→60)
    target_x = track_width * (60 / 400)

    actions = ActionChains(driver)
    actions.click_and_hold(left_handle).move_by_offset(target_x, 0).release().perform()

    print("✔ 상영시간 최소 60분으로 설정 완료!")
    
    # STEP G: 검색 버튼 클릭
    print("STEP G: 검색 버튼 클릭 준비...")

    search_btn = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((
            By.XPATH,
            "//a[contains(@class,'load_more') and contains(text(),'검색')]"
        ))
    )

    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", search_btn) #코드 정렬용. 없어도 되긴하는데 있어야 안정
    time.sleep(0.2)

    search_btn.click()
    print("✔ 검색 버튼 클릭 완료!")

if __name__ == "__main__":
    test_click_sort()


정렬 패널 열림!
드롭다운 열림!
STEP C: 평점 내림차순 선택 중...
평점 내림차순 옵션 찾음 → 클릭!
평점 내림차순 선택 완료!
STEP D: 날짜 입력 시작...
✔ 개봉일 검색 체크박스 활성화
✔ 검색날짜 입력 완료: 2025-01-01
✔ 검색날짜(끝) 입력 완료: 2025-03-31
✔ 날짜 입력 확정 완료!
STEP E: 언어 한국어 선택 시작...
STEP E(1): 언어 드롭다운 열기 시도...
✔ STEP E(1): 언어 드롭다운 열림 완료!
✔ 한국어 항목 발견 → 클릭!
✔ 언어 ‘한국어’ UI 클릭으로 선택 완료!
STEP F: 슬라이더 핸들 드래그 시작...
✔ 상영시간 최소 60분으로 설정 완료!
STEP G: 검색 버튼 클릭 준비...
✔ 검색 버튼 클릭 완료!
